# Data Sources

**Primary Dataset:**  
Jackson Divakar, *Online Shopping Dataset*, Kaggle.

https://www.kaggle.com/datasets/jacksondivakarr/online-shopping-dataset/data

**Secondary Dataset:**  
Peretz Cohen, *2019 Census US Population Data By State*, Kaggle.

https://www.kaggle.com/datasets/peretzcohen/2019-census-us-population-data-by-state

The secondary dataset provides 2019 population estimates together with
latitude and longitude information for the geographic locations represented
in the dataset.

## Hello, Data!

In this step, the raw transaction data is loaded from the CSV file.
The first three records are displayed to confirm that the file was
loaded correctly and to inspect the structure of the data.

In [1]:
import pandas as pd

df = pd.read_csv("../data/sales_data.csv")

df.head(3)

,Unnamed: 0,CustomerID,Gender,Location,Tenure_Months,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,...,Avg_Price,Delivery_Charges,Coupon_Status,GST,Date,Offline_Spend,Online_Spend,Month,Coupon_Code,Discount_pct
0,0,17850.0,M,Chicago,12.0,16679.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,...,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
1,1,17850.0,M,Chicago,12.0,16680.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,...,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
2,2,17850.0,M,Chicago,12.0,16696.0,2019-01-01,GGOENEBQ078999,Nest Cam Outdoor Security Camera - USA,Nest-USA,...,122.77,6.5,Not Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0


In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 52955 entries, 0 to 52954
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           52955 non-null  int64  
 1   CustomerID           52924 non-null  float64
 2   Gender               52924 non-null  str    
 3   Location             52924 non-null  str    
 4   Tenure_Months        52924 non-null  float64
 5   Transaction_ID       52924 non-null  float64
 6   Transaction_Date     52924 non-null  str    
 7   Product_SKU          52924 non-null  str    
 8   Product_Description  52924 non-null  str    
 9   Product_Category     52955 non-null  str    
 10  Quantity             52924 non-null  float64
 11  Avg_Price            52924 non-null  float64
 12  Delivery_Charges     52924 non-null  float64
 13  Coupon_Status        52924 non-null  str    
 14  GST                  52924 non-null  float64
 15  Date                 52924 non-null  str    
 1

## Picking the Right Container


A dictionary works well for a transaction because you can label each field with a clear key like customer_id, price, or quantity, while a set is handy for finding unique values like the different shipping cities in your data. A NamedTuple fits best when you want fixed, ordered fields that you can also access by name, like Transaction(customer_id=1, price=9.99, quantity=2), giving you dictionary-like readability with tuple-like structure.

## Implementing Functions and Data Structure

A simple `Transaction` class is created to represent a transaction.
The class contains basic transaction information and two methods:
`total()` calculates the transaction amount, while `clean()` removes
extra spaces from the product name.


In [3]:
class Transaction:
    def __init__(self, product, price, quantity):
        self.product = product
        self.price = price
        self.quantity = quantity

    def total(self):
        return self.price * self.quantity

    def clean(self):
        # Remove unnecessary whitespace from the product name
        if isinstance(self.product, str):
            self.product = self.product.strip()

        # Convert numeric values to appropriate types
        self.price = float(self.price)
        self.quantity = int(self.quantity)

        return self


transaction = Transaction(
    df.iloc[0]["Product_Description"],
    df.iloc[0]["Avg_Price"],
    df.iloc[0]["Quantity"]
)

transaction.clean()

print("Product:", transaction.product)
print("Price:", transaction.price)
print("Quantity:", transaction.quantity)
print("Total:", transaction.total())

Product: Nest Learning Thermostat 3rd Gen-USA - Stainless Steel
Price: 153.71
Quantity: 1
Total: 153.71


## Step 4: Bulk Loading

The DataFrame contains many transaction records, so processing each row manually
would not be practical. In this step, the transaction records are converted from
the DataFrame into a list of dictionaries.

Each dictionary represents one transaction, with the column names becoming the
dictionary keys.

In [4]:
transactions = df.to_dict(orient="records")

print("Number of transactions:", len(transactions))

# Displaying the first transaction
transactions[0]

Number of transactions: 52955


{'Unnamed: 0': 0,
 'CustomerID': 17850.0,
 'Gender': 'M',
 'Location': 'Chicago',
 'Tenure_Months': 12.0,
 'Transaction_ID': 16679.0,
 'Transaction_Date': '2019-01-01',
 'Product_SKU': 'GGOENEBJ079499',
 'Product_Description': 'Nest Learning Thermostat 3rd Gen-USA - Stainless Steel',
 'Product_Category': 'Nest-USA',
 'Quantity': 1.0,
 'Avg_Price': 153.71,
 'Delivery_Charges': 6.5,
 'Coupon_Status': 'Used',
 'GST': 0.1,
 'Date': '1/1/2019',
 'Offline_Spend': 4500.0,
 'Online_Spend': 2424.5,
 'Month': 1,
 'Coupon_Code': 'ELEC10',
 'Discount_pct': 10.0}

## Quick Profiling

The `describe()` function is used to quickly profile the dataset.
The default version summarizes numerical columns, while
`describe(include='object')` summarizes categorical columns while also showing the count of unique locations

In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,52955.0,26477.000000,15286.936089,0.00,13238.50,26477.00,39715.50,52954.00
CustomerID,52924.0,15346.709810,1766.556020,12346.00,13869.00,15311.00,16996.25,18283.00
Tenure_Months,52924.0,26.127995,13.478285,2.00,15.00,27.00,37.00,50.00
Transaction_ID,52924.0,32409.825675,8648.668977,16679.00,25384.00,32625.50,39126.25,48497.00
Quantity,52924.0,4.497638,20.104711,1.00,1.00,1.00,2.00,900.00
Avg_Price,52924.0,52.237646,64.006882,0.39,5.70,16.99,102.13,355.74
Delivery_Charges,52924.0,10.517630,19.475613,0.00,6.00,6.00,6.50,521.36
GST,52924.0,0.137462,0.045825,0.05,0.10,0.18,0.18,0.18
Offline_Spend,52924.0,2830.914141,936.154247,500.00,2500.00,3000.00,3500.00,5000.00
Online_Spend,52924.0,1893.109119,807.014092,320.25,1252.63,1837.87,2425.35,4556.93


In [6]:
df.describe(include = 'object').T


C:\Users\KING\AppData\Local\Temp\ipykernel_7516\4114222867.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include = 'object').T


,count,unique,top,freq
Gender,52924,2,F,33007
Location,52924,5,Chicago,18380
Transaction_Date,52924,365,2019-11-27,335
Product_SKU,52924,1145,GGOENEBJ079499,3511
Product_Description,52924,404,Nest Learning Thermostat 3rd Gen-USA - Stainle...,3511
Product_Category,52955,21,Apparel,18126
Coupon_Status,52924,3,Clicked,26926
Date,52924,365,11/27/2019,335
Coupon_Code,52555,48,SALE20,6373


I also used simple descriptive statistics to understand the numerical values in the dataset. I also used a Python `set` to count the unique customer locations. The price statistics help identify the typical price range, while the unique-location count gives a quick view of geographic coverage.


In [7]:
print("Minimum price:", df["Avg_Price"].min())
print("Mean price:", df["Avg_Price"].mean())
print("Maximum price:", df["Avg_Price"].max())

unique_cities = set(df["Location"].dropna())
print("Unique cities:", len(unique_cities))
print("Cities:", unique_cities)

Minimum price: 0.39
Mean price: 52.237646436399366
Maximum price: 355.74
Unique cities: 5
Cities: {'Washington DC', 'New York', 'Chicago', 'California', 'New Jersey'}


## Spotting the Grime in the dataset

The profiling results reveal several data-quality problems. First, multiple customer and transaction fields contain missing values. Second, `Coupon_Code` and `Discount_pct` contain missing values. Third, there are extreme numerical values: `Quantity` reaches 900 even though the median quantity is 1, and `Delivery_Charges` reaches more than 500 while the median is 6. Finally, `Unnamed: 0` appears to be an imported index rather than a useful business field.



In [8]:
# Count missing values
missing = df.isna().sum()
print("Missing values:")
print(missing[missing > 0])

# Check unusual numerical values
print("Quantity median:", df["Quantity"].median())
print("Quantity maximum:", df["Quantity"].max())

print("Delivery charge median:", df["Delivery_Charges"].median())
print("Delivery charge maximum:", df["Delivery_Charges"].max())

# Check the redundant index column
print("Unnamed column:", "Unnamed: 0" in df.columns)

Missing values:
CustomerID              31
Gender                  31
Location                31
Tenure_Months           31
Transaction_ID          31
Transaction_Date        31
Product_SKU             31
Product_Description     31
Quantity                31
Avg_Price               31
Delivery_Charges        31
Coupon_Status           31
GST                     31
Date                    31
Offline_Spend           31
Online_Spend            31
Coupon_Code            400
Discount_pct           400
dtype: int64
Quantity median: 1.0
Quantity maximum: 900.0
Delivery charge median: 6.0
Delivery charge maximum: 521.36
Unnamed column: True


## Cleaning Rules

#### I carried out the following:

- Removed Unnamed: 0.
- Converted dates to actual datetime values.
- Removed rows missing essential transaction fields.
- Replaced missing coupon values with "NO_COUPON".
- Replaced missing discount with 0.
- Stripped whitespace from text fields.
- Removed impossible/non-positive quantities.
- Kept the extreme quantity values visible rather than silently deleting them.

In [9]:
print("Rows before cleaning:", len(df))
print("Missing values before cleaning:", df.isna().sum().sum())


cleaned_df = df.copy()


cleaned_df = cleaned_df.drop(columns=["Unnamed: 0"])

# Clean text fields
text_columns = [
    "Gender",
    "Location",
    "Product_Description",
    "Product_Category",
    "Coupon_Status",
    "Coupon_Code"
]

for column in text_columns:
    cleaned_df[column] = cleaned_df[column].astype("string").str.strip()

# Fill optional coupon information
cleaned_df["Coupon_Code"] = cleaned_df["Coupon_Code"].fillna("NO_COUPON")
cleaned_df["Discount_pct"] = cleaned_df["Discount_pct"].fillna(0)

# Convert dates
cleaned_df["Transaction_Date"] = pd.to_datetime(
    cleaned_df["Transaction_Date"],
    errors="coerce"
)

cleaned_df["Date"] = pd.to_datetime(
    cleaned_df["Date"],
    errors="coerce"
)

# Remove rows missing essential transaction information
essential_columns = [
    "CustomerID",
    "Transaction_ID",
    "Transaction_Date",
    "Product_Description",
    "Quantity",
    "Avg_Price"
]

cleaned_df = cleaned_df.dropna(subset=essential_columns)

# Remove invalid quantities and prices
cleaned_df = cleaned_df[
    (cleaned_df["Quantity"] > 0) &
    (cleaned_df["Avg_Price"] >= 0)
].copy()

print("Rows after cleaning:", len(cleaned_df))
print("Missing values after cleaning:", cleaned_df.isna().sum().sum())

Rows before cleaning: 52955
Missing values before cleaning: 1296
Rows after cleaning: 52924
Missing values after cleaning: 0


## Transformations

The coupon information contains both a coupon code and a discount percentage. The dataset already provides `Discount_pct`, but I also create a numeric discount-rate field so that the percentage can be used directly in calculations.


In [10]:
cleaned_df["discount_rate"] = cleaned_df["Discount_pct"] / 100

cleaned_df[
    ["Coupon_Code", "Discount_pct", "discount_rate"]
].head()

,Coupon_Code,Discount_pct,discount_rate
0,ELEC10,10.0,0.1
1,ELEC10,10.0,0.1
2,ELEC10,10.0,0.1
3,ELEC10,10.0,0.1
4,ELEC10,10.0,0.1


### Creating a transaction revenue field

In [11]:
cleaned_df["gross_revenue"] = (
    cleaned_df["Avg_Price"] * cleaned_df["Quantity"]
)

cleaned_df["discount_amount"] = (
    cleaned_df["gross_revenue"] * cleaned_df["discount_rate"]
)

cleaned_df["net_revenue"] = (
    cleaned_df["gross_revenue"]
    - cleaned_df["discount_amount"]
    + cleaned_df["Delivery_Charges"]
)

cleaned_df[
    [
        "Avg_Price",
        "Quantity",
        "Discount_pct",
        "gross_revenue",
        "discount_amount",
        "net_revenue"
    ]
].head()

,Avg_Price,Quantity,Discount_pct,gross_revenue,discount_amount,net_revenue
0,153.71,1.0,10.0,153.71,15.371,144.839
1,153.71,1.0,10.0,153.71,15.371,144.839
2,122.77,2.0,10.0,245.54,24.554,227.486
3,81.50,1.0,10.0,81.50,8.150,79.850
4,153.71,1.0,10.0,153.71,15.371,144.839


## Feature Engineering

I created `days_since_purchase` from the transaction date. It represents the number of days between each transaction date and the chosen reference date of September 22, 2026. A fixed reference date was used so that the results remain reproducible when the notebook is executed at a later date.. I also create a simple `revenue_per_item` feature to describe the average revenue associated with each purchased item.


In [12]:
reference_date = pd.Timestamp("2026-09-22")

cleaned_df["days_since_purchase"] = (
    reference_date - cleaned_df["Transaction_Date"]
).dt.days

cleaned_df["revenue_per_item"] = (
    cleaned_df["net_revenue"] / cleaned_df["Quantity"]
)

cleaned_df[
    [
        "Transaction_Date",
        "days_since_purchase",
        "Quantity",
        "net_revenue",
        "revenue_per_item"
    ]
].head()

,Transaction_Date,days_since_purchase,Quantity,net_revenue,revenue_per_item
0,2019-01-01,2821,1.0,144.839,144.839
1,2019-01-01,2821,1.0,144.839,144.839
2,2019-01-01,2821,2.0,227.486,113.743
3,2019-01-01,2821,1.0,79.850,79.850
4,2019-01-01,2821,1.0,144.839,144.839


## Mini-Aggregation

I grouped the cleaned transactions by `Location` and calculated total net revenue for each location. This provides a simple comparison of revenue across the cities represented in the transaction data.


In [13]:
revenue_by_city = (
    cleaned_df
    .groupby("Location")["net_revenue"]
    .sum()
    .sort_values(ascending=False)
)

revenue_by_city

Location
Chicago          1494006.598
California       1329056.286
New York          872236.464
New Jersey        370933.147
Washington DC     235238.952
Name: net_revenue, dtype: float64

## Serialization Checkpoint

The cleaned and enriched dataset is saved in both CSV and JSON formats. CSV is convenient for tabular data exchange, while JSON preserves each transaction as a structured record.


In [14]:
import os

os.makedirs("../output", exist_ok=True)

cleaned_df.to_csv(
    "../output/cleaned_sales_data.csv",
    index=False
)

cleaned_df.to_json(
    "../output/cleaned_sales_data.json",
    orient="records",
    date_format="iso",
    indent=2
)

print("CSV and JSON files successfully created.")

CSV and JSON files successfully created.


In [15]:
json_check = pd.read_json(
    "../output/cleaned_sales_data.json"
)

print("JSON rows:", len(json_check))
print("JSON columns:", len(json_check.columns))

JSON rows: 52924
JSON columns: 26


## Soft Interview Reflection

Using Functions helped me make the data-processing workflow more organized and easily readable. Instead of writing the same cleaning or calculation logic repeatedly, I could place it inside methods such as `clean()` and `total()`. 

The `Transaction` class also made the relationship between a product, its price, its quantity, and its total value clearer. During this lab, I learnt that functions are especially useful when a task must be applied repeatedly to many records. 

They also make code easier to test, troubleshoot and understand because each function has a specific responsibility.


# Data Dictionary

The data dictionary below documents all fields in the final dataset. It
combines fields from the primary e-commerce transaction dataset, the
secondary 2019 population dataset, and fields derived during the data
cleaning, transformation, and feature engineering stages.

| Field | Type | Description | Source |
|---|---|---|---|
| `CustomerID` | Integer | Unique identifier assigned to each customer. | Primary |
| `Gender` | String | Gender recorded for the customer. | Primary |
| `Location` | String | Geographic location associated with the customer or transaction. | Primary |
| `Tenure_Months` | Integer | Number of months the customer has been associated with the business. | Primary |
| `Transaction_ID` | String | Unique identifier for the transaction. | Primary |
| `Transaction_Date` | Date | Date on which the transaction occurred. | Primary |
| `Product_SKU` | String | Stock keeping unit used to identify the product. | Primary |
| `Product_Description` | String | Description or name of the purchased product. | Primary |
| `Product_Category` | String | Category to which the purchased product belongs. | Primary |
| `Quantity` | Integer | Number of units purchased in the transaction. | Primary |
| `Avg_Price` | Float | Average price per unit of the purchased product. | Primary |
| `Delivery_Charges` | Float | Delivery or shipping charges associated with the transaction. | Primary |
| `Coupon_Status` | String | Status of the coupon associated with the transaction. | Primary |
| `GST` | Float | Goods and Services Tax amount associated with the transaction. | Primary |
| `Date` | Date | Date associated with the corresponding record. | Primary |
| `Offline_Spend` | Float | Amount spent through offline marketing or sales activity. | Primary |
| `Online_Spend` | Float | Amount spent through online marketing or sales activity. | Primary |
| `Month` | String | Month associated with the transaction or marketing record. | Primary |
| `Coupon_Code` | String | Promotional coupon code associated with the transaction. | Primary |
| `Discount_pct` | Float | Percentage discount applied to the transaction. | Primary |
| `population` | Integer | Estimated population for the corresponding geographic location in 2019. | Secondary |
| `latitude` | Float | Latitude coordinate associated with the corresponding geographic location. | Secondary |
| `longitude` | Float | Longitude coordinate associated with the corresponding geographic location. | Secondary |
| `discount_rate` | Float | Discount percentage converted into a decimal rate for calculations. | Derived |
| `gross_revenue` | Float | Revenue before applying the discount, calculated from average price and quantity. | Derived |
| `discount_amount` | Float | Monetary value of the discount applied to the gross revenue. | Derived |
| `net_revenue` | Float | Revenue after applying the discount and accounting for delivery charges. | Derived |
| `days_since_purchase` | Integer | Number of days between the transaction date and the selected reference date. | Derived |
| `revenue_per_item` | Float | Net revenue generated per item purchased. | Derived |

The derived fields were created during the data cleaning, transformation,
and feature engineering stages of this project.

## Secondary Dataset

For the secondary data source, I use the **2019 Census US Population Data By State** dataset from Kaggle.

The secondary dataset provides population and geographic information, including:

- `STATE` — state/location name
- `POPESTIMATE2019` — estimated population in 2019
- `lat` — latitude
- `long` — longitude

This dataset is used as a metadata source to enrich the cleaned e-commerce transaction data.

In [27]:
secondary_dataset = pd.read_csv("../data/2019_Census_US_Population_Data_By_State_Lat_Long.csv")
secondary_dataset.head(3)

,STATE,POPESTIMATE2019,lat,long
0,Alabama,4903185,32.377716,-86.300568
1,Alaska,731545,58.301598,-134.420212
2,Arizona,7278717,33.448143,-112.096962


### Merge Method

The primary `Location` column is matched directly to the secondary `STATE` column.

The following secondary fields are retained:

- `POPESTIMATE2019`
- `lat`
- `long`

After the merge, these fields are renamed to more descriptive names:

- `POPESTIMATE2019` → `population`
- `lat` → `latitude`
- `long` → `longitude`

In [28]:
# Direct match between primary Location and secondary STATE
merged_df = cleaned_df.merge(
    secondary_dataset[
        ["STATE", "POPESTIMATE2019", "lat", "long"]
    ],
    left_on="Location",
    right_on="STATE",
    how="inner"
)

# Rename secondary fields
merged_df = merged_df.rename(columns={
    "POPESTIMATE2019": "population",
    "lat": "latitude",
    "long": "longitude"
})

print("Rows before merge:", len(cleaned_df))
print("Rows after merge:", len(merged_df))
print("Unique matched locations:", merged_df["Location"].nunique())

Rows before merge: 52924
Rows after merge: 52924
Unique matched locations: 5


## Merge Verification

After merging, I compare the number of rows before and after the merge and check the unique locations that were successfully matched.

I also inspect the resulting population and geographic fields to confirm that the secondary metadata was added correctly.

In [31]:
merged_df[
    ["Location", "population", "latitude", "longitude"]
].drop_duplicates()

,Location,population,latitude,longitude
0,Chicago,3565287,41.764046,-72.682198
23,California,39512223,38.576668,-121.493629
29,New York,19453561,42.652843,-73.757874
38,New Jersey,8882190,40.220596,-74.769913
115,Washington DC,7614893,47.035805,-122.905014


In [32]:
ecommerce_analysis_df = merged_df

In [33]:
import os

os.makedirs("../output", exist_ok=True)

cleaned_df.to_csv(
    "../output/Final_ecommerce_analysis_data.csv",
    index=False
)

cleaned_df.to_json(
    "../output/Final_ecommerce_analysis.json",
    orient="records",
    date_format="iso",
    indent=2
)

print("CSV and JSON files successfully created.")

CSV and JSON files successfully created.


## Merge Summary

The cleaned primary transaction data was enriched using the secondary 2019 population dataset.

The merge was performed using an exact match between:

`cleaned_df["Location"]` → `secondary_dataset["STATE"]`

The resulting dataset contains the original cleaned transaction fields together with:

- `population_2019`
- `latitude`
- `longitude`

Only locations that appeared in both datasets were retained through the inner join.